# Análise de Rejeição e Dissonância Cognitiva

Este notebook explora o conceito de **Polarização Afetiva**. Em democracias polarizadas, o "voto negativo" (rejeição absoluta a um candidato ou partido) frequentemente supera a convergência ideológica.

## Objetivos
1. Mapear o índice de Rejeição Absoluta.
2. Aplicar um "Filtro de Veto" ao modelo matemático de recomendação.
3. Identificar Dissonância Cognitiva (quando a recomendação ideal sofre veto do eleitor).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import euclidean_distances

sns.set_theme(style='whitegrid')


## 1. Mapeamento da Rejeição Absoluta
A pergunta "Quem você considera uma ameaça e NÃO votaria sob nenhuma hipótese?" captura essa identidade negativa extrema.


In [ ]:
df = pd.read_csv('../data/raw/mock_voters.csv')

plt.figure(figsize=(10, 5))
sns.countplot(data=df, y='rejeicao_absoluta', order=df['rejeicao_absoluta'].value_counts().index, palette='Reds_r')
plt.title('Índice de Rejeição Absoluta dos Eleitores')
plt.xlabel('Quantidade de Eleitores')
plt.ylabel('Candidato Rejeitado')
plt.show()


## 2. O Paradoxo da Polarização (Dissonância Cognitiva)
Vamos simular o cenário onde identificamos o "Candidato Ideal" de um eleitor matematicamente (menor Distância Euclidiana). A Dissonância Cognitiva ocorre se esse Candidato Ideal for EXATAMENTE a pessoa que o eleitor declarou "Rejeição Absoluta".


In [ ]:
politicos_data = {
    'Político': ['Lula', 'Flavio Bolsonaro', 'Romeu Zema', 'Ciro Gomes'],
    'ideologia': ['Pragmatismo Progressista', 'Direita Radical', 'Neoliberalismo', 'Nacional-Desenvolvimentismo'],
    'justificativa': ['Conciliação de classes e pautas sociais', 'Pauta de costumes conservadora e punitivismo', 'Livre mercado estrito e alianças de conveniência', 'Estado forte na economia e crítica ao rentismo'],
    'bloco_d_estatais': [1, 5, 5, 2],
    'bloco_d_tabelamento': [4, 1, 1, 3],
    'bloco_e_punitivismo': [2, 5, 4, 3],
    'bloco_e_educacao': [2, 5, 4, 2],
    'bloco_g_corrupcao': [1, 5, 5, 2],
    'bloco_g_pesquisa': [5, 1, 2, 5],
    'bloco_g_politica_externa': [5, 1, 2, 4]
}
df_politicos = pd.DataFrame(politicos_data).set_index('Político')
cols_opiniao = ['bloco_d_estatais', 'bloco_d_tabelamento', 'bloco_e_punitivismo', 'bloco_e_educacao', 'bloco_g_corrupcao', 'bloco_g_pesquisa', 'bloco_g_politica_externa']
df_cand = df_politicos.copy()
vetores_cand = df_cand[cols_opiniao].values

# Calcular a recomendação puramente matemática (Sem Filtro) para toda a base
dist_matriz = euclidean_distances(df[cols_opiniao].values, vetores_cand)
df['candidato_matematico_ideal'] = [df_cand.index[i] for i in np.argmin(dist_matriz, axis=1)]

display(df[['candidato_matematico_ideal', 'rejeicao_absoluta']].head(10))


## 3. Aplicando o Filtro de Veto
Se o candidato_matematico_ideal == rejeicao_absoluta, o algoritmo VAA não pode recomendá-lo. Ele deve aplicar uma penalidade (distância infinita) e buscar a 2ª melhor opção.


In [ ]:
def recomendacao_com_veto(row, dist_row):
    rejeicao = row['rejeicao_absoluta']
    
    # Cria uma cópia das distâncias do eleitor para os 4 candidatos
    dist_corrigida = dist_row.copy()
    
    # Se o candidato rejeitado estiver na matriz, aplicamos veto (distância infinita)
    if rejeicao in df_cand.index:
        idx_rejeitado = df_cand.index.get_loc(rejeicao)
        dist_corrigida[idx_rejeitado] = np.inf
        
    # Retorna o índice do menor valor APÓS o veto
    idx_recomendado = np.argmin(dist_corrigida)
    return df_cand.index[idx_recomendado]

# Aplicando a lógica de veto
df['candidato_recomendado_final'] = [recomendacao_com_veto(row, dist_matriz[i]) for i, row in df.iterrows()]

# Calculando quantos sofreram dissonância cognitiva
df['dissonancia'] = df['candidato_matematico_ideal'] != df['candidato_recomendado_final']

taxa_dissonancia = df['dissonancia'].mean() * 100
print(f'{taxa_dissonancia:.2f}% do eleitorado possui uma inclinação política inconsciente para o candidato que eles mesmos rejeitam.')

plt.figure(figsize=(6, 6))
plt.pie(df['dissonancia'].value_counts(), labels=['Voto Consistente', 'Dissonância / Veto Aplicado'], autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'])
plt.title('Impacto da Polarização Afetiva no Recomendador')
plt.show()
